# 10 · Credits, estimates, quotas and typed errors

**Use case:** before wiring Langsat into automation you want to know what things cost, how much is left, and that a
script can branch on *what* went wrong — without parsing English.

**Sub-tasks**
1. Balance and usage (`credits.dashboard`, `credits.usage`, `credits.quota_status`)
2. `estimates()` on the analysis project and on a modeling project (training, serving)
3. The error catalogue: `NotFound`, `Invalid`, and what a `MissingScope` looks like
4. The key's scopes (`me()`) — least privilege

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
d = ls.credits.dashboard()
print("tier:", d.get("tier"), "· balance:", d.get("balance_credits"), "· used this month:", d.get("credits_used_this_month"))
pool = d.get("org_pool") or {}
if pool: print("team pool: balance", pool.get("balance_credits"), "· used this month", pool.get("credits_used_this_month"))
print("quota:", json.dumps(ls.credits.quota_status(), default=str)[:400])

tier: team · balance: 0 · used this month: 0
team pool: balance 999984170 · used this month 20688


quota: {"usage": {"pct_today": 1, "pct_month": 0}, "compute": {"pct_today": 0, "pct_month": 0, "used_today_usd": 0.0139, "used_month_usd": 0.0139, "cap_today_usd": 3.333, "cap_month_usd": 100.0}, "extend": {"enabled": true, "spent_today_usd": 0.0, "spent_month_usd": 0.0}, "active_inference_engines": 0, "inference_engines": [], "tier_config": {"hardware": "gpu", "cr_per_min": 133, "max_duration_min": 60, 


In [3]:
explore = get_or_create_project(ls, "explore", kind="data_analysis")
ds = get_or_create_project(ls, "verified", kind="data_science")
for name, p in (("explore", explore), ("verified (data_science)", ds)):
    est = p.estimates()
    print(f"\n{name}: rows {est['rows_total']}")
    for k in ("clean", "refresh", "train", "ask", "predict", "serving"):
        print(f"  {k:<8} {est.get(k)}")

reusing project 0b6ffd93-08da-4220-9dfa-5b62b5f5951f (amazon-reviews-explore, status=schema_done)
project ready: status=schema_done · type=data_analysis · files=['customer.csv', 'product.csv', 'review.csv']


reusing project 28106708-9ec0-48e4-b292-d600b9bc2ed8 (amazon-reviews-verified, status=ready)


project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']



explore: rows 37325
  clean    {'credits': 0, 'lane': 'lambda'}
  refresh  {'credits': 0, 'lane': 'lambda'}
  train    None
  ask      {'ai_questions': 1, 'quota': {'available': True, 'pct_today': 1, 'pct_month': 0, 'reason': None}}
  predict  {'credits_per_call': 50}
  serving  {'instance_type': 'r5.xlarge', 'ondemand': {'credits_per_hour': 2304, 'credits_per_month_always_on': 1658880}, 'spot': {'credits_per_hour': 922, 'credits_per_month_always_on': 663840}}



verified (data_science): rows 37325
  clean    {'credits': 0, 'lane': 'lambda'}
  refresh  {'credits': 0, 'lane': 'lambda'}
  train    {'instance_type': 'g4dn.2xlarge', 'suggested_minutes': 71, 'embed_minutes_estimate': 8, 'text_cols': 4, 'credits_route': '/api/v1/credits/estimate-training'}
  ask      {'ai_questions': 1, 'quota': {'available': True, 'pct_today': 1, 'pct_month': 0, 'reason': None}}
  predict  {'credits_per_call': 50}
  serving  {'instance_type': 'r5.xlarge', 'ondemand': {'credits_per_hour': 2304, 'credits_per_month_always_on': 1658880}, 'spot': {'credits_per_hour': 922, 'credits_per_month_always_on': 663840}}


## Errors you can branch on

Every 4xx carries `X-Error-Code`; the SDK raises a typed exception with `.code`, `.status`, `.message`.

In [4]:
from langsat import errors
try:
    ls.projects.get("00000000-0000-0000-0000-000000000000")
except errors.NotFound as e:
    print("NotFound ·", e.status, e.code, "·", e.message)
try:
    ls.predict.forecast(project_id=explore.id, target="rating", window="fortnight", length=3)
except errors.Invalid as e:                               # 422: the request shape — detail lists the field
    bad = e.detail[0] if isinstance(e.detail, list) and e.detail else e.detail
    print("Invalid ·", e.status, "·", (bad.get("loc"), bad.get("msg")) if isinstance(bad, dict) else bad)
try:
    explore.hosting.configure_serving(enabled=True)      # needs the hosting:write scope this key does not carry
except errors.MissingScope as e:
    print("MissingScope ·", e.status, e.code, "· needs scope", e.scope, "—", e.message)

NotFound · 404 not_found · Project not found


Invalid · 422 · (['body', 'window_type'], "Input should be 'minute', 'hour', 'day', 'week', 'month' or 'year'")
MissingScope · 403 missing_scope · needs scope hosting:write — This API key does not carry the 'hosting:write' scope.


In [5]:
me = ls.me()
key = me["api_key"]
print("key:", key["name"], "· scopes:", key["scopes"])
print("projects allow-list:", key["project_ids"] or "all", "· expires:", key["expires_at"])
print("\nA key minted with only projects:read + data:read would get `MissingScope` (403, scope='projects:write') on projects.create(...) —")
print("mint keys with the smallest set a script needs: Settings → API keys → Permissions.")

key: TEST_SDK_2 · scopes: ['chat', 'credits:read', 'dashboards:read', 'dashboards:write', 'data:read', 'data:write', 'inference:read', 'keys:read', 'model:read', 'monitoring:read', 'monitoring:write', 'predict', 'projects:read', 'projects:write', 'sources:write', 'teams:read', 'train']
projects allow-list: all · expires: 2026-10-15T02:30:09.291509+00:00

A key minted with only projects:read + data:read would get `MissingScope` (403, scope='projects:write') on projects.create(...) —
mint keys with the smallest set a script needs: Settings → API keys → Permissions.


In [6]:
save_metrics(".", {"notebook": "10_credits_estimates_and_errors", "task": "credits, estimates, typed errors", "model": "—",
                   "headline": {"tier": d.get("tier"), "scopes_on_key": len(key["scopes"]), "errors_demonstrated": ["NotFound", "Invalid/LangsatError", "Conflict/LangsatError"]}})

wrote results/metrics.json


PosixPath('results/metrics.json')